This notebook consolidates four modeling approaches for maternal mortality ratio (MMR) prediction and forecasting. The Linear Regression section defines shared preprocessing used by the Random Forest and XGBoost models to keep comparisons consistent. ARIMA is treated as a time-series model with stationarity checks and forecasts.

# Linear Regression

Baseline model, simple but may underperform compared with non-linear approaches.

## Data Preparation

We load the shared dataset once, clean year columns, and construct a modeling table that includes lagged values and categorical encodings used for tree/boosted models. This keeps the training/test splits consistent across sections.

In [ ]:
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

In [ ]:
DATA_PATH = "Maternal_Mortality.csv"

raw_df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {raw_df.shape}")
raw_df.head()

In [ ]:
year_pattern = re.compile(r"\((19|20)\d{2}\)$")
year_cols = [c for c in raw_df.columns if year_pattern.search(c)]

undp_col = next((c for c in raw_df.columns if "UNDP" in c), None)
if undp_col is None:
    raise ValueError("Could not find UNDP region column in dataset.")

id_cols = [
    "ISO3",
    "Country",
    "Continent",
    "Hemisphere",
    "Human Development Groups",
    undp_col,
    "HDI Rank (2021)",
]

mmr_long = raw_df.melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name="MMR_Year_Column",
    value_name="MMR",
)

mmr_long["Year"] = mmr_long["MMR_Year_Column"].str.extract(
    r"(\d{4})").astype(int)
mmr_long["MMR"] = pd.to_numeric(mmr_long["MMR"], errors="coerce")
mmr_long["HDI Rank (2021)"] = pd.to_numeric(
    mmr_long["HDI Rank (2021)"], errors="coerce")

mmr_long["Human Development Groups"] = mmr_long["Human Development Groups"].fillna(
    "Unknown")
mmr_long["Continent"] = mmr_long["Continent"].fillna("Unknown")
mmr_long[undp_col] = mmr_long[undp_col].fillna("Unknown")
mmr_long["HDI Rank (2021)"] = mmr_long["HDI Rank (2021)"].fillna(
    mmr_long["HDI Rank (2021)"].median()
)

hdi_map = {"Low": 0, "Medium": 1, "High": 2, "Very High": 3}
mmr_long["hdi_group_enc"] = mmr_long["Human Development Groups"].map(
    hdi_map).fillna(1).astype(int)

continent_encoder = LabelEncoder()
undp_encoder = LabelEncoder()
mmr_long["continent_enc"] = continent_encoder.fit_transform(
    mmr_long["Continent"].astype(str))
mmr_long["undp_enc"] = undp_encoder.fit_transform(
    mmr_long[undp_col].astype(str))

mmr_long = mmr_long.sort_values(["ISO3", "Year"]).reset_index(drop=True)

mmr_long["lag_1"] = mmr_long.groupby("ISO3")["MMR"].shift(1)
mmr_long["lag_2"] = mmr_long.groupby("ISO3")["MMR"].shift(2)
mmr_long["lag_3"] = mmr_long.groupby("ISO3")["MMR"].shift(3)
mmr_long["rolling_mean_5"] = (
    mmr_long.groupby("ISO3")["MMR"]
    .shift(1)
    .rolling(window=5, min_periods=1)
    .mean()
)
mmr_long["mmr_change"] = mmr_long.groupby("ISO3")["MMR"].diff()
mmr_long["year_norm"] = (mmr_long["Year"] - mmr_long["Year"].min()) / (
    mmr_long["Year"].max() - mmr_long["Year"].min()
)

model_df = mmr_long.dropna(
    subset=["MMR", "lag_1", "lag_2", "lag_3", "rolling_mean_5", "mmr_change"]
).copy()

print(f"Prepared dataset shape: {model_df.shape}")
model_df[["ISO3", "Country", "Year", "MMR", "lag_1", "rolling_mean_5"]].head()

## Model Building

We fit both a simple and a multi-feature linear regression. The multi-feature variant uses standard scaling so coefficients are comparable across inputs.

In [ ]:
mmr_columns = [c for c in raw_df.columns if "Maternal Mortality" in c]
X_cols = mmr_columns[:-1]
y_col = mmr_columns[-1]

X_simple = raw_df[[X_cols[-1]]].copy()
y_simple = raw_df[y_col].copy()

valid_mask_simple = ~y_simple.isna()
X_simple = X_simple[valid_mask_simple].reset_index(drop=True)
y_simple = y_simple[valid_mask_simple].reset_index(drop=True)

X_train_simple, X_test_simple, y_train_simple, y_test_simple = train_test_split(
    X_simple, y_simple, test_size=0.2, random_state=42
)

lr_simple = LinearRegression()
lr_simple.fit(X_train_simple, y_train_simple)

y_pred_train_simple = lr_simple.predict(X_train_simple)
y_pred_test_simple = lr_simple.predict(X_test_simple)

print(
    f"Simple LR: y = {lr_simple.intercept_:.4f} + {lr_simple.coef_[0]:.4f} * MMR_2020")

X_multi = raw_df[X_cols].copy()
if "HDI Rank (2021)" in raw_df.columns:
    X_multi = pd.concat([
        X_multi,
        raw_df[["HDI Rank (2021)"]].fillna(raw_df["HDI Rank (2021)"].mean())
    ], axis=1)

y_multi = raw_df[y_col].copy()
valid_mask = ~y_multi.isna()
X_multi = X_multi[valid_mask].reset_index(drop=True)
y_multi = y_multi[valid_mask].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_multi = LinearRegression()
lr_multi.fit(X_train_scaled, y_train)

y_pred_train = lr_multi.predict(X_train_scaled)
y_pred_test = lr_multi.predict(X_test_scaled)

feature_importance_lr = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lr_multi.coef_,
    "Abs_Coefficient": np.abs(lr_multi.coef_)
}).sort_values("Abs_Coefficient", ascending=False)

## Model Evaluation

We report MAE, RMSE, and R² for both the simple and multi-feature linear regressions to establish a baseline for comparison.

In [ ]:
simple_rmse_test = np.sqrt(mean_squared_error(
    y_test_simple, y_pred_test_simple))
simple_mae_test = mean_absolute_error(y_test_simple, y_pred_test_simple)
simple_r2_test = r2_score(y_test_simple, y_pred_test_simple)

multi_rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
multi_mae_test = mean_absolute_error(y_test, y_pred_test)
multi_r2_test = r2_score(y_test, y_pred_test)

print("Simple Linear Regression")
print(f"  MAE : {simple_mae_test:.4f}")
print(f"  RMSE: {simple_rmse_test:.4f}")
print(f"  R2  : {simple_r2_test:.4f}\n")

print("Multiple Linear Regression")
print(f"  MAE : {multi_mae_test:.4f}")
print(f"  RMSE: {multi_rmse_test:.4f}")
print(f"  R2  : {multi_r2_test:.4f}")

## Feature Analysis

We visualize the largest positive and negative coefficients from the multi-feature regression to interpret which years contribute most strongly to 2021 outcomes.

In [ ]:
top_features = feature_importance_lr.head(10)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#1D9E75" if val >
          0 else "#D85A30" for val in top_features["Coefficient"]]
ax.barh(top_features["Feature"],
        top_features["Coefficient"], color=colors, alpha=0.8)
ax.set_title("Top Linear Regression Coefficients")
ax.set_xlabel("Coefficient")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

## Data Visualization

We show actual vs predicted plots for both simple and multi-feature models to assess fit and residual patterns.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
min_val = min(y_test_simple.min(), y_pred_test_simple.min())
max_val = max(y_test_simple.max(), y_pred_test_simple.max())
axes[0].scatter(y_test_simple, y_pred_test_simple, alpha=0.6, s=50)
axes[0].plot([min_val, max_val], [min_val, max_val], "r--", lw=2)
axes[0].set_title("Simple LR: Actual vs Predicted")
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")
axes[0].grid(True, alpha=0.3)

min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
axes[1].scatter(y_test, y_pred_test, alpha=0.6, s=50, color="#1D9E75")
axes[1].plot([min_val, max_val], [min_val, max_val], "r--", lw=2)
axes[1].set_title("Multiple LR: Actual vs Predicted")
axes[1].set_xlabel("Actual")
axes[1].set_ylabel("Predicted")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Random Forest

Captures nonlinear relationships and typically improves accuracy over linear baselines.

## Data Preparation

We reuse the shared long-format dataset and precomputed lag features from the Linear Regression section to keep comparisons consistent.

## Model Building

We train a Random Forest regressor using the same core engineered features as XGBoost. A temporal split ensures we evaluate on future years to match forecasting goals.

In [ ]:
rf_feature_cols = [
    "Year",
    "HDI Rank (2021)",
    "lag_1",
    "lag_2",
    "lag_3",
    "rolling_mean_5",
    "mmr_change",
    "year_norm",
    "continent_enc",
    "hdi_group_enc",
    "undp_enc",
]

rf_train_df = model_df[model_df["Year"] <= 2017].copy()
rf_test_df = model_df[model_df["Year"] > 2017].copy()

X_train_rf = rf_train_df[rf_feature_cols]
y_train_rf = rf_train_df["MMR"]
X_test_rf = rf_test_df[rf_feature_cols]
y_test_rf = rf_test_df["MMR"]

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train_rf, y_train_rf)
y_pred_rf = rf_model.predict(X_test_rf)

## Model Evaluation

We report MAE, RMSE, and R² on the held-out future years to compare against the linear baseline and XGBoost.

In [ ]:
rf_mae = mean_absolute_error(y_test_rf, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test_rf, y_pred_rf))
rf_r2 = r2_score(y_test_rf, y_pred_rf)

print("Random Forest Evaluation")
print(f"  MAE : {rf_mae:.4f}")
print(f"  RMSE: {rf_rmse:.4f}")
print(f"  R2  : {rf_r2:.4f}")

## Feature Analysis

Random Forest feature importances highlight which lagged indicators and structural variables drive forecasts most strongly.

In [ ]:
rf_importance = pd.DataFrame({
    "feature": rf_feature_cols,
    "importance": rf_model.feature_importances_,
}).sort_values("importance", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=rf_importance.head(12), x="importance",
            y="feature", palette="crest")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## Data Visualization

We plot the prediction error distribution and compare actual vs predicted values to assess fit.

In [ ]:
residuals_rf = y_test_rf - y_pred_rf

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals_rf, bins=25, color="#7F77DD",
             edgecolor="black", alpha=0.7)
axes[0].set_title("Random Forest Residuals")
axes[0].set_xlabel("Residual")
axes[0].set_ylabel("Frequency")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_test_rf, y_pred_rf, alpha=0.6, s=40)
min_val = min(y_test_rf.min(), y_pred_rf.min())
max_val = max(y_test_rf.max(), y_pred_rf.max())
axes[1].plot([min_val, max_val], [min_val, max_val], "r--", lw=2)
axes[1].set_title("Random Forest: Actual vs Predicted")
axes[1].set_xlabel("Actual")
axes[1].set_ylabel("Predicted")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# XGBoost

Expected best performance due to boosted trees and strong handling of non-linear signals.

## Data Preparation

We reuse the shared long-format dataset, lag features, and categorical encodings to keep a consistent feature space.

## Model Building

We train a gradient-boosted regressor with conservative learning rate and sufficient estimators to reduce bias while controlling variance.

In [ ]:
xgb_feature_cols = rf_feature_cols

xgb_train_df = model_df[model_df["Year"] <= 2017].copy()
xgb_test_df = model_df[model_df["Year"] > 2017].copy()

X_train_xgb = xgb_train_df[xgb_feature_cols]
y_train_xgb = xgb_train_df["MMR"]
X_test_xgb = xgb_test_df[xgb_feature_cols]
y_test_xgb = xgb_test_df["MMR"]

xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror",
)

xgb_model.fit(X_train_xgb, y_train_xgb)
y_pred_xgb = xgb_model.predict(X_test_xgb)

## Model Evaluation

We use MAE, RMSE, and R² to evaluate boosted predictions on the held-out years.

In [ ]:
xgb_mae = mean_absolute_error(y_test_xgb, y_pred_xgb)
xgb_rmse = np.sqrt(mean_squared_error(y_test_xgb, y_pred_xgb))
xgb_r2 = r2_score(y_test_xgb, y_pred_xgb)

print("XGBoost Evaluation")
print(f"  MAE : {xgb_mae:.4f}")
print(f"  RMSE: {xgb_rmse:.4f}")
print(f"  R2  : {xgb_r2:.4f}")

## Feature Analysis

We inspect and plot the learned feature importance values to identify dominant lag features and structural indicators.

In [ ]:
xgb_importance = pd.DataFrame({
    "feature": xgb_feature_cols,
    "importance": xgb_model.feature_importances_,
}).sort_values("importance", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=xgb_importance.head(12), x="importance",
            y="feature", palette="viridis")
plt.title("XGBoost Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## Data Visualization

We visualize predicted vs actual values and the global/continent trend line to maintain consistency with other models.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(x=y_test_xgb, y=y_pred_xgb, alpha=0.5, ax=axes[0])
min_val = min(y_test_xgb.min(), y_pred_xgb.min())
max_val = max(y_test_xgb.max(), y_pred_xgb.max())
axes[0].plot([min_val, max_val], [min_val, max_val], "r--", lw=2)
axes[0].set_title("XGBoost: Actual vs Predicted")
axes[0].set_xlabel("Actual MMR")
axes[0].set_ylabel("Predicted MMR")
axes[0].grid(True, alpha=0.3)

viz_df = mmr_long.dropna(subset=["MMR"]).copy()
global_yearly = viz_df.groupby("Year", as_index=False)["MMR"].mean()
continent_yearly = viz_df.groupby(
    ["Year", "Continent"], as_index=False)["MMR"].mean()

sns.lineplot(data=global_yearly, x="Year",
             y="MMR", ax=axes[1], color="#1D9E75")
axes[1].set_title("Global Average MMR Trend")
axes[1].set_ylabel("Average MMR")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ARIMA

Used for time-series forecasting on a single country trajectory; strong for temporal patterns but does not use multivariate features.

## Data Preparation

We isolate a single country series, convert year strings into a datetime index, and remove missing values to form a clean time series.

In [ ]:
country_name = "Kenya"

country_df = raw_df[raw_df["Country"] == country_name].copy()
if country_df.empty:
    raise ValueError(f"Country '{country_name}' not found in dataset.")

country_ts = country_df[year_cols].T
country_ts.columns = ["MMR"]
country_ts.index = country_ts.index.str.extract(r"(\d{4})")[0]
country_ts.index = pd.to_datetime(country_ts.index)
country_ts["MMR"] = pd.to_numeric(country_ts["MMR"], errors="coerce")
country_ts = country_ts.dropna()

## Model Building

We test stationarity using the Augmented Dickey-Fuller test and then fit a baseline ARIMA(1,1,1) model for multi-step forecasting.

In [ ]:
adf_stat, adf_pvalue, _, _, _, _ = adfuller(country_ts["MMR"])
print(f"ADF Statistic: {adf_stat:.4f}")
print(f"ADF p-value : {adf_pvalue:.4f}")

arima_model = ARIMA(country_ts["MMR"], order=(1, 1, 1))
arima_fit = arima_model.fit()

forecast_steps = 5
forecast = arima_fit.forecast(steps=forecast_steps)
forecast

## Model Evaluation

We report the forecasted values and visually compare historical vs forecasted trends.

In [ ]:
forecast_df = pd.DataFrame({"Forecast": forecast})
forecast_df.index.name = "Year"
forecast_df

## Feature Analysis

ARIMA does not provide feature importances; instead we interpret the time-series parameters and differencing order used in the model.

In [ ]:
print(arima_fit.summary())

## Data Visualization

We plot the historical series and overlay the 5-step forecast to show projected movement.

In [ ]:
forecast_index = pd.date_range(
    start=country_ts.index[-1] + pd.offsets.YearBegin(1),
    periods=forecast_steps,
    freq="YS",
)
forecast_series = pd.Series(forecast.values, index=forecast_index)

plt.figure(figsize=(12, 6))
plt.plot(country_ts.index, country_ts["MMR"], label="Historical", marker="o")
plt.plot(forecast_series.index, forecast_series.values,
         label="Forecast", marker="o")
plt.title(f"ARIMA Forecast for {country_name}")
plt.xlabel("Year")
plt.ylabel("MMR")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()